In [ ]:
import os
import pathlib

import imageio
import matplotlib.pyplot as plt
import natsort
import numpy as np
import pandas as pd
import seaborn as sns
import skimage
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

import imageio.v3 as iio

In [ ]:
# create LUT for each channel
# cyan, magenta, green
cyan_lut = np.zeros((256, 3), dtype=np.uint8)
cyan_lut[:, 0] = np.arange(256)  # red channel
cyan_lut[:, 1] = np.arange(256)  # green channel
cyan_lut[:, 2] = 0  # blue channel
magenta_lut = np.zeros((256, 3), dtype=np.uint8)
magenta_lut[:, 0] = np.arange(256)  # red channel
magenta_lut[:, 1] = 0  # green channel
magenta_lut[:, 2] = np.arange(256)  # blue channel
green_lut = np.zeros((256, 3), dtype=np.uint8)
green_lut[:, 0] = 0  # red channel
green_lut[:, 1] = np.arange(256)  # green channel
green_lut[:, 2] = 0  # blue channel
# make each lut into a a plt.cm colormap
cyan_cmap = plt.cm.colors.ListedColormap(cyan_lut / 255)
magenta_cmap = plt.cm.colors.ListedColormap(magenta_lut / 255)
green_cmap = plt.cm.colors.ListedColormap(green_lut / 255)


def add_scale_bar(
    composite_image,
    pixel_size_microns,
    scale_bar_length_microns,
    scale_bar_height_pixels=10,
    padding_pixels=10,
):
    # Calculate the length of the scale bar in pixels
    scale_bar_length_pixels = int(scale_bar_length_microns / pixel_size_microns)

    # Add the scale bar to the image in the lower right corner
    composite_image[
        -scale_bar_height_pixels - padding_pixels : -padding_pixels,
        -scale_bar_length_pixels - padding_pixels : -padding_pixels,
        :,
    ] = [
        1,
        1,
        1,
    ]  # white scale bar with a padding pixel margin from the edges
    return composite_image


def create_composite_image(cyan_channel, magenta_channel, green_channel):
    """Create a composite Cyan-Magenta-Green image from the individual channels."""
    vmin1, vmax1 = stretch(cyan_channel, low=2, high=98)
    vmin2, vmax2 = stretch(magenta_channel, low=20, high=99)
    vmin3, vmax3 = stretch(green_channel, low=2, high=98)
    # normalize the channels to the range [0, 255] using the vmin and vmax values
    cyan_channel = np.clip(
        (cyan_channel - vmin1) / (vmax1 - vmin1) * 255, 0, 255
    ).astype(np.uint8)
    magenta_channel = np.clip(
        (magenta_channel - vmin2) / (vmax2 - vmin2) * 255, 0, 255
    ).astype(np.uint8)
    green_channel = np.clip(
        (green_channel - vmin3) / (vmax3 - vmin3) * 255, 0, 255
    ).astype(np.uint8)
    # apply the LUTs to each channel
    cyan_rgb = cyan_cmap(cyan_channel)[:, :, :3]  # drop alpha channel
    magenta_rgb = magenta_cmap(magenta_channel)[:, :, :3]
    green_rgb = green_cmap(green_channel)[:, :, :3]
    # combine the channels by taking the maximum value across the three channels for each pixel
    composite_rgb = np.maximum(np.maximum(cyan_rgb, magenta_rgb), green_rgb)
    return composite_rgb


def create_sytox_green_video(sytox_green_channel):
    """Create a video of the SYTOX Green channel with a consistent stretch across frames."""
    vmin, vmax = stretch(sytox_green_channel, low=15, high=99)
    # normalize the channel to the range [0, 255] using the vmin and vmax values
    sytox_green_channel = np.clip(
        (sytox_green_channel - vmin) / (vmax - vmin) * 255, 0, 255
    ).astype(np.uint8)
    # apply the green LUT to the channel
    sytox_green_rgb = green_cmap(sytox_green_channel)[:, :, :3]  # drop alpha channel
    return sytox_green_rgb


def stretch(img, low=2, high=98):
    """Percentile-based contrast stretch — returns (vmin, vmax)."""
    return np.percentile(img, (low, high))

In [ ]:
# set the paths to the data and output directories
image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(f"{image_base_dir}/processed_data/").resolve(strict=True)

processed_image_dir = pathlib.Path(
    f"{image_base_dir}/1.illumination_corrected_files/"
).resolve(strict=True)

platemap_file_path = pathlib.Path(
    f"{root_dir}/Wave2_data/0.download_data/platemap/platemap.csv"
).resolve(strict=True)
platemap_df = pd.read_csv(platemap_file_path)
platemap_df["unique_condition"] = (
    platemap_df["Metadata_Inducer"]
    + "_"
    + platemap_df["Metadata_Inducer_dose"].astype(str)
    + "_"
    + platemap_df["Metadata_Inhibitor"]
    + "_"
    + platemap_df["Metadata_Inhibitor_dose"].astype(str)
)

In [4]:
ic_file_path = pathlib.Path(
    f"{image_base_dir}/1.illumination_corrected_files/"
).resolve(strict=True)
out_dict = {
    "file_path": [],
}
for well_fov in tqdm.tqdm(ic_file_path.glob("*")):
    for image_file_path in well_fov.glob("*.tiff"):

        out_dict["file_path"].append(image_file_path)
out_df = pd.DataFrame(out_dict)
out_df.insert(0, "file_name", out_df["file_path"].apply(lambda x: x.name))
out_df.insert(1, "well", out_df["file_name"].apply(lambda x: x.split("_")[0]))
out_df.insert(2, "fov", out_df["file_name"].apply(lambda x: x.split("_")[1]))
out_df.insert(3, "time", out_df["file_name"].apply(lambda x: x.split("_")[2]))
out_df.insert(
    4, "channel", out_df["file_name"].apply(lambda x: x.split("_")[3].split(".")[0])
)
out_df.sort_values(by=["well", "fov", "time", "channel"], inplace=True)
# pivot wide the channel column so that we have one row per well/fov/time and columns for each channel
out_df = out_df.pivot_table(
    index=["well", "fov", "time"],
    columns="channel",
    values="file_path",
    aggfunc="first",
).reset_index()

out_df.columns.name = None  # clean up the "channel" label on the column axis

KeyboardInterrupt: 

In [ ]:
df = pd.merge(
    out_df,
    platemap_df,
    left_on="well",
    right_on="Metadata_Well",
    how="left",
)
# natsort byt well fov time
df = df.sort_values(
    by=["well", "fov", "time"], key=lambda x: x.map(lambda y: natsort.natsort_key(y))
).reset_index(drop=True)
df.head()

In [ ]:
# subset to one well fov
well_fov = "B2_1"
df_subset = df[
    (df["well"] == well_fov.split("_")[0]) & (df["fov"] == well_fov.split("_")[1])
].reset_index(drop=True)
# load in the images for the first time point

The images were acquired in 5 channels:
| Channel # | Channel Name | Excitation (nm) | Emission (nm) | Dye | Exposure (ms) |
|:---------:|:------------:|:----------:|:--------:|:------:|:---:|
| 1 | ChromaLIVE | 640 | 685/40 | ChromaLIVE | 50 |
| 2 | ChromaLIVE | 488 | 685/40 | ChromaLIVE | 200 |
| 3 | SYTOX Green | 488 | 525/50 | SYTOX Green | 50 |
| 4 | Nuclei | 405 | 447/60 | NulceoLive Blue | 50 |
| 5 | Brightfield | | | | 50 |


In [ ]:
C1, C2, C3, C4, C5 = (
    "ChromaLIVE640",
    "ChromaLIVE488",
    "SYTOX Green",
    "NucleoLIVE",
    "Brightfield",
)
i1, i2, i3, i4, i5 = (
    # read using bioio
    skimage.io.imread(df_subset.loc[0, "C1"]),
    skimage.io.imread(df_subset.loc[0, "C2"]),
    skimage.io.imread(df_subset.loc[0, "C3"]),
    skimage.io.imread(df_subset.loc[0, "C4"]),
    skimage.io.imread(df_subset.loc[0, "C5"]),
)
plt.figure(figsize=(20, 4))
vmin1, vmax1 = stretch(i1, low=2, high=98)
vmin2, vmax2 = stretch(i2, low=15, high=99)
vmin3, vmax3 = stretch(i3, low=15, high=99)
vmin4, vmax4 = stretch(i4, low=2, high=98)
vmin5, vmax5 = stretch(i5, low=2, high=98)

plt.subplot(1, 5, 1)
plt.imshow(i1, cmap=green_cmap, vmin=vmin1, vmax=vmax1)
plt.title(C1)
plt.axis("off")
plt.subplot(1, 5, 2)
plt.imshow(i2, cmap=magenta_cmap, vmin=vmin2, vmax=vmax2)
plt.title(C2)
plt.axis("off")
plt.subplot(1, 5, 3)
plt.imshow(i3, cmap=green_cmap, vmin=vmin3, vmax=vmax3)
plt.title(C3)
plt.axis("off")
plt.subplot(1, 5, 4)
plt.imshow(i4, cmap=cyan_cmap, vmin=vmin4, vmax=vmax4)
plt.title(C4)
plt.axis("off")
plt.subplot(1, 5, 5)
plt.imshow(i5, cmap="gray", vmin=vmin5, vmax=vmax5)
plt.title(C5)
plt.axis("off")
plt.suptitle(
    f"Well {well_fov.split('_')[0]} FOV {well_fov.split('_')[1]} Time {df_subset.loc[0, 'time']}"
)
plt.show()

In [ ]:
# loop through each well fov and create a video of the composite images across time points
df["well_fov"] = df["well"] + "_" + df["fov"]
for condition in tqdm.tqdm(df["unique_condition"].unique()):
    df_subset = df[df["unique_condition"] == condition].reset_index(drop=True)
    # pick the first well fov for this condition
    well_fov = df_subset["well_fov"].unique()[0]
    unique_condition = df_subset["unique_condition"].unique()[0]
    output_video_path = pathlib.Path(
        f"{root_dir}/Wave2_data/figures/montages/videos/{well_fov}_{unique_condition}_composite_video.mp4"
    ).resolve()
    sytox_green_video_path = pathlib.Path(
        f"{root_dir}/Wave2_data/figures/montages/videos/{well_fov}_{unique_condition}_sytox_green_video.mp4"
    ).resolve()
    output_video_path.parent.mkdir(parents=True, exist_ok=True)
    if output_video_path.exists() and sytox_green_video_path.exists():
        continue
    composite_images = []
    sytox_green_images = []
    for idx, row in df_subset.iterrows():
        green = skimage.io.imread(row["C1"])
        magenta = skimage.io.imread(row["C2"])
        cyan = skimage.io.imread(row["C4"])
        composite_image = create_composite_image(
            cyan_channel=cyan, magenta_channel=magenta, green_channel=green
        )
        composite_image = add_scale_bar(
            composite_image, pixel_size_microns=0.325, scale_bar_length_microns=10
        )
        composite_images.append(composite_image)
        sytox_green = skimage.io.imread(row["C3"])
        sytox_green_image = create_sytox_green_video(sytox_green)
        sytox_green_image = add_scale_bar(
            sytox_green_image, pixel_size_microns=0.325, scale_bar_length_microns=10
        )
        sytox_green_images.append(sytox_green_image)

    # save the composite images as a video using skimage.io.mimwrite
    composite_images_uint8 = [
        (img * 255).astype(np.uint8) if img.dtype != np.uint8 else img
        for img in composite_images
    ]
    sytox_green_image_uint8 = [
        (img * 255).astype(np.uint8) if img.dtype != np.uint8 else img
        for img in sytox_green_images
    ]

    iio.imwrite(
        output_video_path,
        composite_images_uint8,
        plugin="pyav",  # or "ffmpeg" if pyav not available
        fps=3,
        codec="h264",
    )
    iio.imwrite(
        sytox_green_video_path,
        sytox_green_image_uint8,
        plugin="pyav",  # or "ffmpeg" if pyav not available
        fps=3,
        codec="h264",
    )

In [ ]:
# convert all mp4 to gifs
for mp4_path in tqdm.tqdm(
    pathlib.Path(f"{root_dir}/Wave2_data/figures/montages/videos/").glob("*.mp4")
):
    gif_path = mp4_path.with_suffix(".gif")
    gif_path = pathlib.Path(str(gif_path).replace("videos", "gifs"))
    gif_path.parent.mkdir(parents=True, exist_ok=True)
    if gif_path.exists():
        continue
    frames = iio.imread(mp4_path, plugin="pyav")  # read mp4 with pyav
    iio.imwrite(gif_path, frames, plugin="pillow", duration=1000 // 3, loop=0)
    # duration is ms per frame (1000ms / 3fps = 333ms), loop=0 means infinite loop